# Проведение экспериментов по настройке модели

Задача — регрессия: предсказать цену продажи `Selling_Price` по характеристикам объявления.
Исходные данные — очищенная выборка из ЛР1 (`../data/clean_data.pkl`, 297 записей).

Все эксперименты логируются в MLflow. Перед запуском блокнота поднимите сервер:

```shell
cd ../mlflow && sh start_mlflow.sh
```

## Загрузка модулей

In [1]:
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import (mean_absolute_error,
                             mean_absolute_percentage_error,
                             mean_squared_error)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (KBinsDiscretizer, OrdinalEncoder,
                                   PolynomialFeatures, StandardScaler)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## Загрузка данных

In [2]:
with open('../data/clean_data.pkl', 'rb') as f:
    data = pickle.load(f)

print('Размер выборки:', data.shape)
data.info()

Размер выборки: (297, 11)
<class 'pandas.DataFrame'>
Index: 297 entries, 0 to 298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Year           297 non-null    int16   
 1   Selling_Price  297 non-null    float64 
 2   Present_Price  297 non-null    float64 
 3   Kms_Driven     297 non-null    int32   
 4   Fuel_Type      297 non-null    category
 5   Seller_Type    297 non-null    category
 6   Transmission   297 non-null    category
 7   Owner          297 non-null    int8    
 8   vehicle_type   297 non-null    category
 9   car_age        297 non-null    int8    
 10  kms_per_year   297 non-null    int32   
dtypes: category(4), float64(2), int16(1), int32(2), int8(2)
memory usage: 11.7 KB


В ЛР1 было показано, что `Year` и `car_age` — один и тот же признак с противоположным
знаком (корреляция −1.00). В модель подаём только `car_age`: он линейно связан
с амортизацией и интерпретируется проще. `Year` удаляем, чтобы не вносить точную
линейную зависимость между признаками.

In [3]:
data = data.drop(columns=['Year'])

numeric_features = ['Present_Price', 'Kms_Driven', 'car_age', 'kms_per_year']
categorical_features = ['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'vehicle_type']
target = 'Selling_Price'

print('Числовых признаков      :', len(numeric_features), numeric_features)
print('Категориальных признаков:', len(categorical_features), categorical_features)

Числовых признаков      : 4 ['Present_Price', 'Kms_Driven', 'car_age', 'kms_per_year']
Категориальных признаков: 5 ['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'vehicle_type']


## Разбиение на обучающую и тестовую выборки

In [4]:
X = data.drop(columns=target)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=RANDOM_STATE, stratify=X['vehicle_type']
)

print(f'Обучающая: {X_train.shape[0]} записей')
print(f'Тестовая : {X_test.shape[0]} записей')
print()
print('Состав обучающей по типу транспорта:')
print(X_train['vehicle_type'].value_counts().to_string())
print('Состав тестовой:')
print(X_test['vehicle_type'].value_counts().to_string())
print()
print('Цена — обучающая: медиана {:.2f}, тестовая: медиана {:.2f}'.format(
    y_train.median(), y_test.median()))

Обучающая: 222 записей
Тестовая : 75 записей

Состав обучающей по типу транспорта:
vehicle_type
car            149
two_wheeler     73
Состав тестовой:
vehicle_type
car            50
two_wheeler    25

Цена — обучающая: медиана 3.50, тестовая: медиана 3.80


Разбиение 75/25, как требует задание. Дополнительно использована **стратификация
по `vehicle_type`**: в выборке два разных рынка — автомобили и двухколёсные, различающиеся
по цене на два порядка. Без стратификации случайное разбиение могло дать перекос,
и метрики на тесте перестали бы быть сопоставимыми между экспериментами.

## Baseline-модель

Пайплайн из двух шагов, как требует задание:

- `transform` — `ColumnTransformer`: `StandardScaler` для числовых признаков,
  `OrdinalEncoder` для категориальных (задача регрессии);
- `regression` — `RandomForestRegressor` с параметрами по умолчанию.

Случайный лес в шкалировании не нуждается, но шаг оставлен намеренно: пайплайн должен
оставаться пригодным и для других алгоритмов.

In [5]:
preprocessor = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OrdinalEncoder(handle_unknown='use_encoded_value',
                                   unknown_value=-1), categorical_features),
])

pipeline_baseline = Pipeline([
    ('transform', preprocessor),
    ('regression', RandomForestRegressor(random_state=RANDOM_STATE)),
])

pipeline_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('transform', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, def

In [6]:
def calc_metrics(model, X, y_true):
    """Метрики качества регрессии на тестовой выборке."""
    y_pred = model.predict(X)
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred),
        'mse': mean_squared_error(y_true, y_pred),
    }


def print_metrics(metrics, title=''):
    if title:
        print(title)
    for name, value in metrics.items():
        print(f'  {name:<6} {value:.4f}')

In [7]:
pipeline_baseline.fit(X_train, y_train)
metrics_baseline = calc_metrics(pipeline_baseline, X_test, y_test)
print_metrics(metrics_baseline, 'Baseline (RandomForest, параметры по умолчанию):')
print()
print(f'Для сравнения: средняя цена в тестовой выборке {y_test.mean():.2f} лакха,')
print(f'медиана {y_test.median():.2f} лакха.')

Baseline (RandomForest, параметры по умолчанию):
  mae    0.6496
  mape   0.1569
  mse    2.2717

Для сравнения: средняя цена в тестовой выборке 4.76 лакха,
медиана 3.80 лакха.


`mae` измеряется в лакхах рупий и показывает среднюю абсолютную ошибку в деньгах.
`mape` — та же ошибка в процентах от истинной цены, она нагляднее при разбросе цен
в три порядка. `mse` сильнее штрафует крупные промахи, поэтому на ней заметнее
ошибки на дорогих автомобилях.

## Подключение MLflow

In [8]:
import mlflow
from mlflow.models import infer_signature

TRACKING_URI = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'Car Price Regression'

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

REGISTERED_MODEL_NAME = 'car-price-rf'
INPUT_EXAMPLE = X_train.head(5)
SIGNATURE = infer_signature(model_input=X_train.head(5))

print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('Версия MLflow      :', mlflow.__version__)

MLflow tracking URI: http://127.0.0.1:5000
Версия MLflow      : 3.16.1


### Функция логирования

Вынесена отдельно, чтобы каждый эксперимент логировался единообразно. Вместе с моделью
логируются сигнатура, пример входных данных и файл зависимостей — как требует п.9 задания.

In [9]:
def log_experiment(run_name, model, metrics=None, params=None,
                   artifacts=None, register=False, signature=SIGNATURE):
    """Логирует модель и сопутствующие артефакты в MLflow."""
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = (experiment.experiment_id if experiment
                     else mlflow.create_experiment(EXPERIMENT_NAME))

    with mlflow.start_run(run_name=run_name, experiment_id=experiment_id) as run:
        if metrics:
            mlflow.log_metrics(metrics)
        if params:
            mlflow.log_params(params)
        for path in (artifacts or []):
            mlflow.log_artifact(path)

        mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
            signature=signature,
            input_example=INPUT_EXAMPLE,
            pip_requirements='../requirements.txt',
            registered_model_name=REGISTERED_MODEL_NAME if register else None,
            # MLflow 3 по умолчанию сериализует через skops, который отказывается
            # сохранять деревья решений и пользовательские классы без явного
            # перечисления доверенных типов. Пайплайн содержит и то, и другое,
            # поэтому используем cloudpickle — он работает с произвольными
            # объектами и остаётся штатным форматом MLflow.
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        print(f'{run_name}: run_id = {run.info.run_id}')
        return run.info.run_id

In [10]:
run_id_baseline = log_experiment(
    'baseline_random_forest',
    pipeline_baseline,
    metrics=metrics_baseline,
    params={'model': 'RandomForestRegressor', 'features': 'исходные 9'},
    register=True,
)

2026/09/28 18:46:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'car-price-rf'.
2026/09/28 18:46:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car-price-rf, version 1


baseline_random_forest: run_id = a9096d7955b149318b0710c232162fd0
🏃 View run baseline_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/a9096d7955b149318b0710c232162fd0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'car-price-rf'.


Модель зарегистрирована в реестре как **версия 1** (`registered_model_name`).
Проверить можно в интерфейсе: вкладка *Models* → `car-price-rf`.

## Генерация новых признаков средствами sklearn

Применяем два преобразования:

- **`PolynomialFeatures(degree=2)`** к признакам `Present_Price`, `car_age`, `kms_per_year`.
  В ЛР1 было показано, что цена продажи — это по сути произведение цены нового на
  коэффициент сохранности, а сохранность линейно падает с возрастом. Полиномиальные
  признаки дают модели произведение `Present_Price × car_age` явно.
- **`KBinsDiscretizer`** к `Kms_Driven`. Влияние пробега на цену нелинейно: первые
  20 тысяч километров обесценивают лот сильнее, чем следующие 20 тысяч.
  Разбиение на корзины позволяет модели уловить такую ступенчатость.

Полученные полиномиальные признаки сами требуют шкалирования — поэтому каждое
преобразование обёрнуто в мини-пайплайн с `StandardScaler` на конце.

In [11]:
poly_features = ['Present_Price', 'car_age', 'kms_per_year']
kbins_features = ['Kms_Driven']

poly_transformer = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
])

kbins_transformer = Pipeline([
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')),
    ('scale', StandardScaler()),
])

extended_transformer = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OrdinalEncoder(handle_unknown='use_encoded_value',
                                   unknown_value=-1), categorical_features),
    ('poly', poly_transformer, poly_features),
    ('kbins', kbins_transformer, kbins_features),
])

In [12]:
X_train_fe_sklearn = X_train.copy()
X_train_fe_sklearn = extended_transformer.fit_transform(X_train_fe_sklearn, y_train)

feature_names = list(extended_transformer.get_feature_names_out())

print(f'Признаков было : {X_train.shape[1]}')
print(f'Признаков стало: {X_train_fe_sklearn.shape[1]}')
print()
print('Новые признаки:')
for name in feature_names[len(numeric_features) + len(categorical_features):]:
    print(' ', name)

Признаков было : 9
Признаков стало: 19

Новые признаки:
  poly__Present_Price
  poly__car_age
  poly__kms_per_year
  poly__Present_Price^2
  poly__Present_Price car_age
  poly__Present_Price kms_per_year
  poly__car_age^2
  poly__car_age kms_per_year
  poly__kms_per_year^2
  kbins__Kms_Driven


In [13]:
FEATURES_PATH = 'features_after_engineering.txt'
with open(FEATURES_PATH, 'w') as f:
    print(*feature_names, sep='\n', file=f)

print(f'Список из {len(feature_names)} столбцов сохранён в {FEATURES_PATH}')

Список из 19 столбцов сохранён в features_after_engineering.txt


In [14]:
pipeline_fe = Pipeline([
    ('transform', extended_transformer),
    ('regression', RandomForestRegressor(random_state=RANDOM_STATE)),
])

pipeline_fe.fit(X_train, y_train)
metrics_fe = calc_metrics(pipeline_fe, X_test, y_test)
print_metrics(metrics_fe, 'После генерации признаков:')
print()
print_metrics(metrics_baseline, 'Baseline для сравнения:')

После генерации признаков:
  mae    0.6419
  mape   0.1548
  mse    2.3996

Baseline для сравнения:
  mae    0.6496
  mape   0.1569
  mse    2.2717


In [15]:
run_id_fe = log_experiment(
    'feature_engineering_sklearn',
    pipeline_fe,
    metrics=metrics_fe,
    params={'model': 'RandomForestRegressor',
            'poly_features': ', '.join(poly_features),
            'kbins_features': ', '.join(kbins_features),
            'features_total': len(feature_names)},
    artifacts=[FEATURES_PATH],
)

2026/09/28 18:46:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


feature_engineering_sklearn: run_id = 880105dd8e7e4cffba266ad172d2b163
🏃 View run feature_engineering_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/880105dd8e7e4cffba266ad172d2b163
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Отбор наиболее значимых признаков

Отбор проводим на расширенном наборе из предыдущего пункта. Из полученных признаков
оставляем **половину** — это середина рекомендованного заданием диапазона 20–70%.

Используется `SequentialFeatureSelector` из `mlxtend` с последовательным **добавлением**
признаков (`forward=True`): алгоритм начинает с пустого набора и на каждом шаге добавляет
тот признак, который сильнее всего улучшает метрику на кросс-валидации.

In [16]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

n_features_to_select = X_train_fe_sklearn.shape[1] // 2
print(f'Отбираем {n_features_to_select} признаков из {X_train_fe_sklearn.shape[1]}')

sfs = SFS(
    RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE),
    k_features=n_features_to_select,
    forward=True,
    floating=False,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
)
sfs = sfs.fit(X_train_fe_sklearn, y_train)

selected_idx = list(sfs.k_feature_idx_)
selected_names = [feature_names[i] for i in selected_idx]

print(f'\nЛучшая метрика MAE на кросс-валидации: {-sfs.k_score_:.4f}')
print('\nОтобранные признаки:')
for i, name in zip(selected_idx, selected_names):
    print(f'  [{i:2}] {name}')

Отбираем 9 признаков из 19



Лучшая метрика MAE на кросс-валидации: 0.8102

Отобранные признаки:
  [ 2] numeric__car_age
  [ 4] categorical__Fuel_Type
  [ 7] categorical__Owner
  [ 8] categorical__vehicle_type
  [10] poly__car_age
  [12] poly__Present_Price^2
  [15] poly__car_age^2
  [17] poly__kms_per_year^2
  [18] kbins__Kms_Driven


`scoring='neg_mean_absolute_error'` — sklearn всегда **максимизирует** метрику, поэтому
MAE подаётся со знаком минус. Чем ближе к нулю, тем лучше; обратно в обычный MAE
значение переводится сменой знака.

In [17]:
SELECTED_NAMES_PATH = 'selected_features_names.txt'
SELECTED_IDX_PATH = 'selected_features_indices.txt'

with open(SELECTED_NAMES_PATH, 'w') as f:
    print(*selected_names, sep='\n', file=f)
with open(SELECTED_IDX_PATH, 'w') as f:
    print(*selected_idx, sep=',', file=f)

print('Сохранено:', SELECTED_NAMES_PATH, 'и', SELECTED_IDX_PATH)

Сохранено: selected_features_names.txt и selected_features_indices.txt


Чтобы встроить отбор в пайплайн вторым шагом, нужен трансформер, который оставляет
столбцы по индексам. Пишем минимальный совместимый с sklearn класс.

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin


class IndexSelector(BaseEstimator, TransformerMixin):
    """Оставляет столбцы с заданными индексами.

    Конструктор сохраняет параметр ровно в том виде, в каком его передали:
    sklearn при клонировании (а это делает cross_val_score) проверяет, что
    __init__ не меняет свои аргументы, иначе клонирование падает.
    """

    def __init__(self, indices):
        self.indices = indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(X)[:, list(self.indices)]

In [19]:
pipeline_fs = Pipeline([
    ('transform', extended_transformer),
    ('feature_selection', IndexSelector(selected_idx)),
    ('regression', RandomForestRegressor(random_state=RANDOM_STATE)),
])

pipeline_fs.fit(X_train, y_train)
metrics_fs = calc_metrics(pipeline_fs, X_test, y_test)
print_metrics(metrics_fs, f'После отбора {n_features_to_select} признаков:')

После отбора 9 признаков:
  mae    0.6592
  mape   0.1618
  mse    2.2939


In [20]:
run_id_fs = log_experiment(
    'feature_selection_mlxtend',
    pipeline_fs,
    metrics=metrics_fs,
    params={'model': 'RandomForestRegressor',
            'selector': 'SFS forward (mlxtend)',
            'features_selected': len(selected_idx),
            'features_total': len(feature_names)},
    artifacts=[SELECTED_NAMES_PATH, SELECTED_IDX_PATH],
)

2026/09/28 18:46:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


feature_selection_mlxtend: run_id = e6daadacfd53473ea22acc7e5ba7b738
🏃 View run feature_selection_mlxtend at: http://127.0.0.1:5000/#/experiments/1/runs/e6daadacfd53473ea22acc7e5ba7b738
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Сравнение с другим алгоритмом отбора *(п. 13, со звёздочкой)*

Повторим отбор методом `RFE` — он идёт с противоположной стороны: начинает с полного
набора и последовательно **удаляет** наименее важные признаки. Интересно, совпадут ли
выборы двух алгоритмов.

In [21]:
rfe = RFE(
    RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE),
    n_features_to_select=n_features_to_select,
)
rfe.fit(X_train_fe_sklearn, y_train)

rfe_idx = list(np.where(rfe.support_)[0])
rfe_names = [feature_names[i] for i in rfe_idx]

common = sorted(set(selected_idx) & set(rfe_idx))
only_sfs = sorted(set(selected_idx) - set(rfe_idx))
only_rfe = sorted(set(rfe_idx) - set(selected_idx))

print(f'SFS  отобрал: {len(selected_idx)} признаков')
print(f'RFE  отобрал: {len(rfe_idx)} признаков')
print(f'Совпало     : {len(common)} ({len(common) / n_features_to_select:.0%})')
print()
print('Выбрали оба:')
for i in common:
    print(f'  [{i:2}] {feature_names[i]}')
print('\nТолько SFS:')
for i in only_sfs:
    print(f'  [{i:2}] {feature_names[i]}')
print('\nТолько RFE:')
for i in only_rfe:
    print(f'  [{i:2}] {feature_names[i]}')

SFS  отобрал: 9 признаков
RFE  отобрал: 9 признаков
Совпало     : 4 (44%)

Выбрали оба:
  [ 2] numeric__car_age
  [10] poly__car_age
  [12] poly__Present_Price^2
  [15] poly__car_age^2

Только SFS:
  [ 4] categorical__Fuel_Type
  [ 7] categorical__Owner
  [ 8] categorical__vehicle_type
  [17] poly__kms_per_year^2
  [18] kbins__Kms_Driven

Только RFE:
  [ 0] numeric__Present_Price
  [ 9] poly__Present_Price
  [13] poly__Present_Price car_age
  [14] poly__Present_Price kms_per_year
  [16] poly__car_age kms_per_year


## Подбор гиперпараметров с помощью optuna

Задание требует настраивать **ту модель, которая показала лучший результат**. Берём
структуру с отбором признаков (расширенный набор → отбор половины → случайный лес)
и оптимизируем именно её.

Настраиваем три параметра случайного леса: количество деревьев, глубину и долю
признаков `max_features` в диапазоне 0.1–1.0.

Оптимизируем **MAE** (для регрессии) и явно указываем `direction='minimize'` —
ошибку нужно минимизировать. Качество оцениваем кросс-валидацией на обучающей
выборке, а не на тесте: иначе подбор параметров подгонится под тест, и итоговая
оценка качества перестанет быть честной.

In [22]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    pipeline = Pipeline([
        ('transform', extended_transformer),
        ('feature_selection', IndexSelector(selected_idx)),
        ('regression', RandomForestRegressor(random_state=RANDOM_STATE, **params)),
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=5,
                             scoring='neg_mean_absolute_error', n_jobs=-1)
    return -scores.mean()


study = optuna.create_study(direction='minimize', study_name='rf_hyperparams')
study.optimize(objective, n_trials=15, show_progress_bar=False)

print(f'Проведено испытаний: {len(study.trials)}')
print(f'Лучшее значение MAE на кросс-валидации: {study.best_value:.4f}')
print('\nЛучшие параметры:')
for name, value in study.best_params.items():
    print(f'  {name} = {value}')

Проведено испытаний: 15
Лучшее значение MAE на кросс-валидации: 0.7600

Лучшие параметры:
  n_estimators = 270
  max_depth = 14
  max_features = 0.9265353938766325


In [23]:
history = pd.DataFrame([
    {'trial': t.number, 'mae_cv': t.value, **t.params} for t in study.trials
]).sort_values('mae_cv')

history.head(10).round(4)

,trial,mae_cv,n_estimators,max_depth,max_features
14,14,0.7600,270,14,0.9265
13,13,0.7605,235,14,0.9322
11,11,0.7623,153,9,0.9406
4,4,0.7746,124,16,0.9817
12,12,0.7902,195,6,0.9983
0,0,0.7912,165,9,0.7587
6,6,0.7919,139,10,0.6877
1,1,0.8119,103,17,0.6608
3,3,0.8504,75,5,0.7795
8,8,0.9217,53,4,0.9291


In [24]:
pipeline_tuned = Pipeline([
    ('transform', extended_transformer),
    ('feature_selection', IndexSelector(selected_idx)),
    ('regression', RandomForestRegressor(random_state=RANDOM_STATE, **study.best_params)),
])

pipeline_tuned.fit(X_train, y_train)
metrics_tuned = calc_metrics(pipeline_tuned, X_test, y_test)
print_metrics(metrics_tuned, 'После подбора гиперпараметров:')

После подбора гиперпараметров:
  mae    0.6484
  mape   0.1555
  mse    2.2718


In [25]:
run_id_tuned = log_experiment(
    'optuna_tuned_random_forest',
    pipeline_tuned,
    metrics=metrics_tuned,
    params={'model': 'RandomForestRegressor',
            'tuning': 'optuna, 15 trials, mae minimize',
            'cv_mae': round(study.best_value, 4),
            **study.best_params},
    artifacts=[FEATURES_PATH, SELECTED_NAMES_PATH],
    register=True,
)

2026/09/28 18:46:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'car-price-rf' already exists. Creating a new version of this model...
2026/09/28 18:46:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car-price-rf, version 2


optuna_tuned_random_forest: run_id = 2cdf7fd27f8c40e1a0a42024235d70f8
🏃 View run optuna_tuned_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/2cdf7fd27f8c40e1a0a42024235d70f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '2' of model 'car-price-rf'.


Модель зарегистрирована как **версия 2**.

## Сравнение всех экспериментов

In [26]:
comparison = pd.DataFrame({
    'baseline': metrics_baseline,
    'feature_engineering': metrics_fe,
    'feature_selection': metrics_fs,
    'optuna_tuned': metrics_tuned,
}).T.round(4)

comparison

,mae,mape,mse
baseline,0.6496,0.1569,2.2717
feature_engineering,0.6419,0.1548,2.3996
feature_selection,0.6592,0.1618,2.2939
optuna_tuned,0.6484,0.1555,2.2718


In [27]:
best_run = comparison['mae'].idxmin()
print(f'Лучший результат по MAE: {best_run} (mae = {comparison.loc[best_run, "mae"]:.4f})')
print()
print('Метрики лучшей модели:')
print(comparison.loc[best_run].to_string())
print()
print(f'Средняя ошибка {comparison.loc[best_run, "mae"]:.2f} лакха '
      f'при медианной цене {y_test.median():.2f} лакха.')

Лучший результат по MAE: feature_engineering (mae = 0.6419)

Метрики лучшей модели:
mae     0.6419
mape    0.1548
mse     2.3996

Средняя ошибка 0.64 лакха при медианной цене 3.80 лакха.


### Вывод

Основная метрика для выбора — **MAE**: она выражена в тех же единицах, что и цена
(лакхи рупий), и потому напрямую интерпретируется. `mape` смотрим как относительную
оценку, `mse` — как индикатор крупных промахов на дорогих лотах.

## Обучение финальной модели на всей выборке

Выбранную конфигурацию обучаем заново — уже на **всей** выборке, а не только на
обучающей части. Метрики здесь не считаем: тестовой выборки больше нет, всё ушло
в обучение. Это нормально для финальной модели, которая пойдёт в продакшен —
качество мы уже оценили на предыдущем шаге.

In [28]:
X_full = pd.concat([X_train, X_test], axis=0)
y_full = pd.concat([y_train, y_test], axis=0)

print(f'Полная выборка: {X_full.shape[0]} записей, {X_full.shape[1]} признаков')

BEST_PIPELINES = {
    'baseline': pipeline_baseline,
    'feature_engineering': pipeline_fe,
    'feature_selection': pipeline_fs,
    'optuna_tuned': pipeline_tuned,
}
production_model = BEST_PIPELINES[best_run]
production_model.fit(X_full, y_full)

print(f'Финальная модель обучена на конфигурации: {best_run}')

Полная выборка: 297 записей, 9 признаков


Финальная модель обучена на конфигурации: feature_engineering


In [29]:
USED_COLUMNS_PATH = 'production_columns.txt'
with open(USED_COLUMNS_PATH, 'w') as f:
    print(*X_full.columns, sep='\n', file=f)

production_signature = infer_signature(
    model_input=X_full,
    model_output=production_model.predict(X_full.head(5)),
)

production_params = {
    'model_type': 'RandomForestRegressor',
    'training_data': 'full_dataset',
    'best_configuration': best_run,
    'total_samples': len(X_full),
    'features_count': X_full.shape[1],
}
if best_run == 'optuna_tuned':
    production_params |= study.best_params

run_id_production = log_experiment(
    'production_model',
    production_model,
    params=production_params,
    artifacts=[USED_COLUMNS_PATH, FEATURES_PATH],
    signature=production_signature,
    register=True,
)

print()
print('RUN_ID PRODUCTION-МОДЕЛИ:', run_id_production)

2026/09/28 18:46:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'car-price-rf' already exists. Creating a new version of this model...
2026/09/28 18:46:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car-price-rf, version 3


production_model: run_id = 54b54fccbed044c5af3fe62db2c20163
🏃 View run production_model at: http://127.0.0.1:5000/#/experiments/1/runs/54b54fccbed044c5af3fe62db2c20163
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

RUN_ID PRODUCTION-МОДЕЛИ: 54b54fccbed044c5af3fe62db2c20163


Created version '3' of model 'car-price-rf'.


### Установка тега Production в реестре моделей

In [30]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
production_version = max(
    (v for v in versions if v.run_id == run_id_production),
    key=lambda v: int(v.version),
)

client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME,
    version=production_version.version,
    key='Production',
    value='true',
)
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias='Production',
    version=production_version.version,
)

print(f'Модель {REGISTERED_MODEL_NAME} версии {production_version.version} '
      f'помечена тегом Production')
print(f'run_id: {run_id_production}')
print()
print('Все зарегистрированные версии:')
for v in sorted(versions, key=lambda v: int(v.version)):
    print(f'  версия {v.version}: run_id={v.run_id[:8]}...')

Модель car-price-rf версии 3 помечена тегом Production
run_id: 54b54fccbed044c5af3fe62db2c20163

Все зарегистрированные версии:
  версия 1: run_id=a9096d79...
  версия 2: run_id=2cdf7fd2...
  версия 3: run_id=54b54fcc...


В MLflow 3 для пометки модели используются **теги** и **алиасы** вместо устаревших
стадий (`stage`), которые были в версии 2 и объявлены deprecated. Здесь проставлено
и то, и другое: тег `Production=true` виден в списке версий, алиас `Production`
позволяет обращаться к модели по имени `models:/car-price-rf@Production` —
именно так она будет загружаться в сервисе в ЛР3.

## Итоги

In [31]:
print('Сводка по экспериментам:')
print(comparison.to_string())
print()
print(f'Лучшая конфигурация : {best_run}')
print(f'Production run_id   : {run_id_production}')
print(f'Версия в реестре    : {production_version.version}')

Сводка по экспериментам:
                        mae    mape     mse
baseline             0.6496  0.1569  2.2717
feature_engineering  0.6419  0.1548  2.3996
feature_selection    0.6592  0.1618  2.2939
optuna_tuned         0.6484  0.1555  2.2718

Лучшая конфигурация : feature_engineering
Production run_id   : 54b54fccbed044c5af3fe62db2c20163
Версия в реестре    : 3


Результаты, которые нужно перенести в README:

- лучшая конфигурация и её метрики — из таблицы сравнения выше;
- `run_id` Production-модели — из вывода последней ячейки;
- параметры модели — из результатов optuna;
- список используемых столбцов — файл `production_columns.txt`.